# FreeFine Final Geometry — FULL 5,677 Generation · Shard 1/4

This is a **fresh final run**, not a reconstruction from old experimental outputs.

It produces three shard-local result sets:
- `baseline`: fresh original FreeFine
- `SGR_EPSREC`: Move→RING4, Rotate→baseline, Resize non-severe→RING8, Resize severe→EPSREC+prompt
- `SGR_MIDHF_EPSREC`: same common branches, severe resize→MIDHF+EPSREC+prompt

The common branches are computed once from the **fresh baseline generated in this same run** and written into both final pipelines. This is exact common-subexpression reuse; it does not import old candidate images.

**Inputs:** `freefine-sample-metadata`, `geobench2d-coarse-img`, `geobench2d-metrics-subset`, `freefine-geobench2d-bggen`.

**Kaggle:** T4×2, Internet ON. Optional `HF_TOKEN` secret.

If the notebook ends `PARTIAL FINAL SHARD`, save the version, attach that saved notebook output as an input, and rerun this same shard. It will resume exact missing images only.


In [1]:

# ===== 1. Clone exact FreeFine source + clock =====
import os,time,subprocess,glob,json,shutil,csv,hashlib,math,socket,re
from collections import defaultdict,Counter

NB_START=time.time()
COMMIT="4c9fdb971572b32edbeac13464659274c28decbb"
subprocess.run(
    f"mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && "
    f"git clone -q https://github.com/CIawevy/FreeFine.git && "
    f"cd FreeFine && git checkout -q {COMMIT}",
    shell=True,check=True
)
print("✓ cloned + pinned FreeFine",COMMIT[:14])


✓ cloned + pinned FreeFine 4c9fdb971572b3


In [2]:

%%bash
# ===== 2. Generation environment =====
set -e
pip install -q --root-user-action=ignore uv
uv python install 3.10.13
V=/kaggle/temp/freefine_env
PY=$V/bin/python
rm -rf "$V"
uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" "torch==2.1.1" "torchvision==0.16.1" --index-url https://download.pytorch.org/whl/cu121
cd /kaggle/temp/FreeFine
uv pip install --python "$PY" -r requirements.txt || {
  grep -v '^xformers' requirements.txt >/tmp/r.txt
  uv pip install --python "$PY" -r /tmp/r.txt
  uv pip install --python "$PY" xformers
}
uv pip install --python "$PY" einops==0.7.0 omegaconf==2.3.0 "setuptools<70"
"$PY" -c "import torch,diffusers,xformers; print('✓ freefine_env',torch.__version__)"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 53.7 MB/s eta 0:00:00
✓ freefine_env 2.1.1+cu121


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Installed Python 3.10.13 in 1.66s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/freefine_env
Activate with: source /kaggle/temp/freefine_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/freefine_env
Resolved 18 packages in 624ms
 Downloaded networkx
 Downloaded torchvision
 Downloaded pillow
 Downloaded sympy
 Downloaded numpy
 Downloaded triton
 Downloaded torch
Prepared 18 packages in 23.09s
Installed 18 packages in 388ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.32.3
 + fsspec==2026.7.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.3.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.1+cu121
 + torchvision==0.16.1+cu121
 + triton==2.1.0
 + typing-extensions==4.16.0
 + urllib3==1.26.13
Using Python 3.10.13 environment at: /kaggle/temp/freefine_en

In [3]:

# ===== 3. Parametrize released inference script; no algorithmic change yet =====
import os,re,py_compile,pathlib
P="/kaggle/temp/FreeFine/evaluation/FreeFine"
M="/kaggle/temp/FreeFine/src/demo/model.py"

# Historical start_layer control patch (default remains 10).
g=open(M).read()
if not re.search(r'^\s*import os\b',g,re.M):
    g="import os\n"+g
assert "list(range(10, 16))" in g
g=g.replace("list(range(10, 16))","list(range(int(os.environ.get('FF_START_LAYER','10')), 16))")
open(M,"w").write(g)

src=open(f"{P}/freefine_batch_infer_2d.py").read()
src=src.replace("sys.path.append('/data/Hszhu/FreeFine')","sys.path.append('/kaggle/temp/FreeFine')")
src=src.replace(
    'pretrained_model_path = "/data/Hszhu/prompt-to-prompt/stable-diffusion-v1-5/"',
    'pretrained_model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"'
)
old=('        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n'
     '        obj_label = ""\n'
     '        ori_mask = read_and_resize_mask(ori_mask_path)\n')
new=('        ori_mask = read_and_resize_mask(ori_mask_path)\n'
     '        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n'
     '        obj_label = (case.get("obj_label","") if os.environ.get("FF_USE_PROMPT")=="1" else "")\n')
assert old in src
src=src.replace(old,new,1)
src=src.replace('"guidance_scale": 7.5,','"guidance_scale": float(os.environ.get("FF_GUIDANCE","7.5")),')
src=src.replace('"start_step": 35,','"start_step": int(os.environ.get("FF_START_STEP","35")),')
src=src.replace(
    'dataset_json = osp.join(dst_base, "annotations_2d.json")',
    'dataset_json = os.environ.get("FF_SUBSET_JSON", osp.join(dst_base,"annotations_2d.json"))'
)
src=src.replace(
    'dst_gen_dir = osp.join(dst_base, "Geo-Bench-2D/Gen_results_FreeFine_2d")',
    'dst_gen_dir = os.environ.get("FF_OUT_DIR", osp.join(dst_base,"Geo-Bench-2D/Gen_results_FreeFine_2d"))'
)
src=src.replace(
    'base_dir = "/data/Hszhu/dataset/GeoBenchMeta/"',
    'base_dir = "/kaggle/temp/GeoBenchMeta"'
)
open(f"{P}/freefine_sweep_2d.py","w").write(src)
py_compile.compile(f"{P}/freefine_sweep_2d.py",doraise=True)
print("✓ inference script parametrized")


✓ inference script parametrized


In [4]:

# ===== FINAL B PATCH: generic task switches + APG + HFF combos + translation locks =====
import os, re, py_compile

S="/kaggle/temp/FreeFine/evaluation/FreeFine/freefine_sweep_2d.py"
M="/kaggle/temp/FreeFine/src/demo/model.py"
A="/kaggle/temp/FreeFine/src/utils/attention.py"

s=open(S).read()
m=open(M).read()
a=open(A).read()

def replace_once(text, old, new, name):
    assert old in text, f"ANCHOR NOT FOUND: {name}"
    assert new not in text, f"ALREADY PATCHED: {name}"
    return text.replace(old,new,1)

# ------------------------------------------------------------
# 1) Generic per-case task switches. A job can be universal or task-aware.
# ------------------------------------------------------------
router_old="        edit_param = case['edit_param']"
router_new='''        edit_param = case['edit_param']
        _dx,_dy,_dz,_rx,_ry,_rz,_sx,_sy,_sz = edit_param
        if abs(float(_rz))>1e-6:
            _etype='rotate'
        elif abs(float(_sx)-1)>1e-6 or abs(float(_sy)-1)>1e-6:
            _etype='resize'
        else:
            _etype='move'
        os.environ['FF_CASE_TYPE']=_etype
        os.environ['FF_CASE_RZ']=str(float(_rz))

        for _k in ('FF_PRESERVE','FF_PRESERVE_W','FF_USE_PROMPT','FF_GUIDANCE',
                   'FF_HFF_ACTIVE','FF_APG_ACTIVE','FF_RING_ACTIVE',
                   'FF_AA_ACTIVE','FF_EXACT_SHIFT_ACTIVE'):
            os.environ.pop(_k,None)

        if os.environ.get('FF_ROUTER','0')=='1':
            if _etype=='move':
                os.environ['FF_PRESERVE']='1'
                os.environ['FF_PRESERVE_W']='0.3'
            elif _etype=='resize':
                os.environ['FF_USE_PROMPT']='1'
                os.environ['FF_GUIDANCE']='10'

        def _has(_name):
            _types={x.strip() for x in os.environ.get(_name,'').split(',') if x.strip()}
            return _etype in _types

        if _has('FF_PROMPT_TYPES'):
            os.environ['FF_USE_PROMPT']='1'
            os.environ['FF_GUIDANCE']=os.environ.get('FF_PROMPT_GUIDANCE','10')
        if _has('FF_PRESERVE_TYPES'):
            os.environ['FF_PRESERVE']='1'
            os.environ['FF_PRESERVE_W']=os.environ.get('FF_PRESERVE_GLOBAL_W','0.3')

        os.environ['FF_HFF_ACTIVE']='1' if os.environ.get('FF_HFF','0')=='1' and _has('FF_HFF_TYPES') else '0'
        os.environ['FF_APG_ACTIVE']='1' if os.environ.get('FF_APG','0')=='1' and _has('FF_APG_TYPES') else '0'
        os.environ['FF_RING_ACTIVE']='1' if os.environ.get('FF_RING','0')=='1' and _has('FF_RING_TYPES') else '0'
        os.environ['FF_AA_ACTIVE']='1' if os.environ.get('FF_AA_WARP','0')=='1' and _has('FF_AA_TYPES') else '0'
        os.environ['FF_EXACT_SHIFT_ACTIVE']='1' if os.environ.get('FF_EXACT_SHIFT','0')=='1' and _has('FF_EXACT_SHIFT_TYPES') else '0'
'''
s=replace_once(s,router_old,router_new,"generic per-case switches")

# ------------------------------------------------------------
# 2) Translation-aware exact integer shift / AA warp.
# ------------------------------------------------------------
warp_old = "    transformed_image = cv2.warpAffine(src_img, rotation_matrix, (width, height))\n    transformed_mask = cv2.warpAffine(src_mask.astype(np.uint8), rotation_matrix, (width, height),\n                                      flags=cv2.INTER_NEAREST).astype(bool)"
warp_new = '''    _exact = (
        os.environ.get('FF_EXACT_SHIFT_ACTIVE','0')=='1'
        and abs(float(rotation_angle))<1e-6
        and abs(float(resize_scale[0])-1.0)<1e-6
        and abs(float(resize_scale[1])-1.0)<1e-6
        and abs(float(dx)-round(float(dx)))<1e-6
        and abs(float(dy)-round(float(dy)))<1e-6
    )
    if _exact:
        _ix,_iy=int(round(float(dx))),int(round(float(dy)))
        transformed_image=np.zeros_like(src_img)
        transformed_mask=np.zeros_like(src_mask,dtype=bool)
        _xs0=max(0,-_ix); _xs1=min(width,width-_ix)
        _ys0=max(0,-_iy); _ys1=min(height,height-_iy)
        _xd0=_xs0+_ix; _xd1=_xs1+_ix
        _yd0=_ys0+_iy; _yd1=_ys1+_iy
        if _xs1>_xs0 and _ys1>_ys0:
            transformed_image[_yd0:_yd1,_xd0:_xd1]=src_img[_ys0:_ys1,_xs0:_xs1]
            transformed_mask[_yd0:_yd1,_xd0:_xd1]=src_mask[_ys0:_ys1,_xs0:_xs1].astype(bool)
    elif os.environ.get('FF_AA_ACTIVE','0')=='1':
        _S=2
        _M=rotation_matrix.copy()
        _M[:,2]*=_S
        _up=cv2.resize(src_img,(width*_S,height*_S),interpolation=cv2.INTER_LANCZOS4)
        _w=cv2.warpAffine(_up,_M,(width*_S,height*_S),flags=cv2.INTER_LANCZOS4)
        transformed_image=cv2.resize(_w,(width,height),interpolation=cv2.INTER_AREA)
        transformed_mask=cv2.warpAffine(
            src_mask.astype(np.uint8),rotation_matrix,(width,height),
            flags=cv2.INTER_NEAREST
        ).astype(bool)
    else:
        transformed_image = cv2.warpAffine(src_img, rotation_matrix, (width, height))
        transformed_mask = cv2.warpAffine(
            src_mask.astype(np.uint8), rotation_matrix, (width, height),
            flags=cv2.INTER_NEAREST
        ).astype(bool)'''
s=replace_once(s,warp_old,warp_new,"exact/AA warp")

# ------------------------------------------------------------
# 3) Final-output interior lock: keep only a refinement boundary ring.
# This is explicitly post-refinement locking, not a claim that UNet skipped the interior.
# ------------------------------------------------------------
gen_old="        generated_results = model.FreeFine_generation(**params)"
gen_new='''        generated_results = model.FreeFine_generation(**params)
        if os.environ.get('FF_RING_ACTIVE','0')=='1':
            _rw=max(1,int(os.environ.get('FF_RING_WIDTH','8')))
            _tm=(target_mask>127).astype(np.uint8)
            _ker=np.ones((2*_rw+1,2*_rw+1),np.uint8)
            _interior=cv2.erode(_tm,_ker,iterations=1).astype(bool)
            _g=np.asarray(generated_results)
            if _g.ndim==4 and _g.shape[0]==1:
                _g=_g[0]
            if _g.dtype!=np.uint8:
                if _g.max()<=1.5:
                    _g=np.clip(_g*255.0,0,255).astype(np.uint8)
                else:
                    _g=np.clip(_g,0,255).astype(np.uint8)
            generated_results=np.where(_interior[:,:,None],coarse_input.astype(np.uint8),_g)'''
s=replace_once(s,gen_old,gen_new,"boundary-ring output lock")

# ------------------------------------------------------------
# 4) Preservation (same code as historical R1 / B4).
# ------------------------------------------------------------
pres_old=(
"                    latents = self.ctrl_step(noise_pred, t, latents, local_var_reg, eta=eta)[0]\n"
"                latents_list.append(latents)"
)
pres_new=(
"                    latents = self.ctrl_step(noise_pred, t, latents, local_var_reg, eta=eta)[0]\n"
"                if os.environ.get('FF_PRESERVE')=='1' and latents.shape[0]==2:\n"
"                    _cl = refer_latents[i - start_step + 1][0]\n"
"                    _w = float(os.environ.get('FF_PRESERVE_W','0.3'))\n"
"                    _m = local_var_reg[0].to(latents.dtype) if local_var_reg.dim()==4 else local_var_reg.to(latents.dtype)\n"
"                    latents[0] = latents[0]*(1 - _w*_m) + _cl*(_w*_m)\n"
"                latents_list.append(latents)"
)
assert pres_old in m, "preservation anchor missing"
m=m.replace(pres_old,pres_new,1)

# ------------------------------------------------------------
# 5) Minimal HFF implementation for combo arms: hf / midhf.
# ------------------------------------------------------------
sig_old="        last_up_block_idx: int = None,\n    ):"
sig_new='''        last_up_block_idx: int = None,
        ff_feature_ref = None,
        ff_feature_mask = None,
        ff_feature_beta: float = 0.0,
        ff_feature_mode: str = "hf",
        ff_feature_radius: int = 3,
        ff_feature_mid_low: int = 1,
        ff_feature_mid_high: int = 3,
        ff_feature_high_weight: float = 0.65,
        ff_feature_block: int = 1,
    ):'''
a=replace_once(a,sig_old,sig_new,"combo HFF signature")

_tok="all_intermediate_features.append(sample)"
_pos=[x.start() for x in re.finditer(re.escape(_tok),a)]
assert len(_pos)==1
_tp=_pos[0]; _ls=a.rfind("\n",0,_tp)+1; _indent=a[_ls:_tp]
_lines=[
"# Combo HFF: edit streams only.",
"if ff_feature_ref is not None and i == int(ff_feature_block) and float(ff_feature_beta)>0:",
"    _ref=ff_feature_ref.to(device=sample.device,dtype=sample.dtype)",
"    if ff_feature_mask is None:",
"        _mask=torch.ones((sample.shape[0],1,*sample.shape[-2:]),device=sample.device,dtype=sample.dtype)",
"    else:",
"        _mask=ff_feature_mask",
"        if _mask.dim()==2: _mask=_mask[None,None]",
"        elif _mask.dim()==3: _mask=_mask[:,None]",
"        _mask=F.interpolate(_mask.float(),size=sample.shape[-2:],mode='bilinear',align_corners=False).to(device=sample.device,dtype=sample.dtype)",
"        if _mask.shape[0]!=sample.shape[0]: _mask=_mask[:1].expand(sample.shape[0],-1,-1,-1)",
"    _gate=torch.zeros((sample.shape[0],1,1,1),device=sample.device,dtype=sample.dtype)",
"    if sample.shape[0]>=4: _gate[0]=1; _gate[2]=1",
"    else: _gate[0]=1",
"    _mask=_mask*_gate",
"    def _sel(_x):",
"        _xf=torch.fft.fftshift(torch.fft.fft2(_x.float(),dim=(-2,-1)),dim=(-2,-1))",
"        _h,_w=_x.shape[-2:]; _yy=torch.arange(_h,device=_x.device)[:,None]-_h//2; _xx=torch.arange(_w,device=_x.device)[None,:]-_w//2",
"        _rho=torch.sqrt(_yy.float()**2+_xx.float()**2)",
"        _mode=str(ff_feature_mode).lower()",
"        if _mode=='hf': _weight=(_rho>float(ff_feature_radius)).float()",
"        elif _mode=='midhf':",
"            _mid=((_rho>float(ff_feature_mid_low))&(_rho<=float(ff_feature_mid_high))).float()",
"            _high=(_rho>float(ff_feature_mid_high)).float()*float(ff_feature_high_weight)",
"            _weight=_mid+_high",
"        else: raise ValueError(f'combo HFF mode={_mode}')",
"        _y=torch.fft.ifft2(torch.fft.ifftshift(_xf*_weight[None,None],dim=(-2,-1)),dim=(-2,-1)).real",
"        return _y.to(dtype=_x.dtype)",
"    _delta=_sel(_ref)-_sel(sample)",
"    sample=sample+float(ff_feature_beta)*_mask*_delta",
]
a=a[:_ls]+''.join(_indent+x+'\n' for x in _lines)+a[_ls:]

# ------------------------------------------------------------
# 6) Inject same-timestep HFF reference and exact APG.
# ------------------------------------------------------------
assert "    def forward_sampling(" in m and "    def prox_regularization" in m
pre,rest=m.split("    def forward_sampling(",1)
body,post=rest.split("    def prox_regularization",1)

start_anchor="        start_step = num_inference_steps - num_actual_inference_steps\n"
assert start_anchor in body
body=body.replace(start_anchor,start_anchor+"        _apg_running = None\n        _apg_do_log = not getattr(self, '_ff_apg_telemetry_used', False)\n",1)

# Robustly replace only the UNet call inside forward_sampling.
# This avoids depending on the exact blank-line layout around controller.log_mask.
_unet_pat=re.compile(
    r"(?ms)^(?P<indent>[ \t]+)noise_pred[ \t]*=[ \t]*self\.unet\(\s*"
    r"model_inputs\s*,\s*t\s*,\s*encoder_hidden_states\s*=\s*text_embeddings\s*\)\s*$"
)
_um=list(_unet_pat.finditer(body))
if len(_um)!=1:
    _cands=[ln for ln in body.splitlines() if 'noise_pred' in ln and 'self.unet' in ln]
    raise AssertionError(f"forward_sampling UNet call count={len(_um)}; candidates={_cands[:8]}")
_um=_um[0]
noise_new="            _ff_ref=None\n            _ff_beta=0.0\n            if os.environ.get('FF_HFF_ACTIVE','0')=='1':\n                _denom=max(1.0,float((num_inference_steps-1)-start_step))\n                _progress=float(i-start_step)/_denom\n                _horizon=float(os.environ.get('FF_HFF_HORIZON','0.55'))\n                _beta0=float(os.environ.get('FF_HFF_BETA','0.30'))\n                if _horizon>0 and _progress<=_horizon:\n                    _ff_beta=_beta0*0.5*(1.0+np.cos(np.pi*_progress/_horizon))\n                if _ff_beta>1e-8:\n                    _ri=i-start_step+1\n                    _coarse_lat=refer_latents[_ri][0:1]\n                    _src_lat=refer_latents[_ri][1:2]\n                    _ref_pair=torch.cat([_coarse_lat,_src_lat],dim=0)\n                    _ref_inputs=torch.cat([_ref_pair]*2,dim=0)\n                    _saved_att=getattr(self.controller,'cur_att_layer',None)\n                    _saved_step=getattr(self.controller,'cur_step',None)\n                    _ff_list=self.unet(\n                        _ref_inputs,t,encoder_hidden_states=text_embeddings,\n                        last_up_block_idx=int(os.environ.get('FF_HFF_BLOCK','1'))\n                    )\n                    if _saved_att is not None: self.controller.cur_att_layer=_saved_att\n                    if _saved_step is not None: self.controller.cur_step=_saved_step\n                    _ff_ref=_ff_list[-1].detach()\n\n            noise_pred=self.unet(\n                model_inputs,t,encoder_hidden_states=text_embeddings,\n                ff_feature_ref=_ff_ref,\n                ff_feature_mask=getattr(self.controller,'fg_retain_mask_st2',None),\n                ff_feature_beta=_ff_beta,\n                ff_feature_mode=os.environ.get('FF_HFF_MODE','hf'),\n                ff_feature_radius=int(os.environ.get('FF_HFF_RADIUS','3')),\n                ff_feature_mid_low=int(os.environ.get('FF_HFF_MID_LOW','1')),\n                ff_feature_mid_high=int(os.environ.get('FF_HFF_MID_HIGH','3')),\n                ff_feature_high_weight=float(os.environ.get('FF_HFF_HIGH_WEIGHT','0.65')),\n                ff_feature_block=int(os.environ.get('FF_HFF_BLOCK','1')),\n            )"
# Re-indent the replacement to the exact indentation of the matched UNet call.
# The original call lives inside `with torch.no_grad():` (normally 16 spaces);
# hard-coding 12 spaces prematurely exits that block and makes the following
# noise_pred_uncon/noise_pred_con line an unexpected indent.
_noise_indent=_um.group("indent")
_noise_lines=noise_new.splitlines()
_noise_non=[ln for ln in _noise_lines if ln.strip()]
_noise_min=min(len(ln)-len(ln.lstrip(" ")) for ln in _noise_non)
noise_new="\n".join(
    (_noise_indent+ln[_noise_min:]) if ln.strip() else ""
    for ln in _noise_lines
)
print(f"✓ replacement re-indented to {len(_noise_indent)} spaces")
body=body[:_um.start()]+noise_new+body[_um.end():]
print("✓ forward_sampling UNet call patched via regex")

_guide_pat=re.compile(
    r"(?ms)"
    r"^(?P<indent>[ \t]+)if[ \t]+not[ \t]+local_edit_text:[ \t]*\n"
    r"(?P=indent)[ \t]+noise_pred[ \t]*=[ \t]*noise_pred_uncon[ \t]*\+[ \t]*guidance_scale[ \t]*\*[ \t]*\(noise_pred_con[ \t]*-[ \t]*noise_pred_uncon\)[ \t]*\n"
    r"(?P=indent)else:[ \t]*\n"
    r"(?P=indent)[ \t]+local_text_guidance[ \t]*=[ \t]*guidance_scale[ \t]*\*[ \t]*\(noise_pred_con[ \t]*-[ \t]*noise_pred_uncon\)[ \t]*\*[ \t]*completion_mask_cfg[ \t]*\n"
    r"(?P=indent)[ \t]+noise_pred[ \t]*=[ \t]*noise_pred_uncon[ \t]*\+[ \t]*local_text_guidance[ \t]*$"
)
_gm=list(_guide_pat.finditer(body))
assert len(_gm)==1, f"APG block count={len(_gm)}"
_gm=_gm[0]; _indent=_gm.group("indent")

guide_new=r'''            _gmode=(
                os.environ.get('FF_GUIDANCE_MODE','apg_exact').lower()
                if os.environ.get('FF_APG_ACTIVE','0')=='1'
                else 'cfg'
            )

            if _gmode=='apg_exact':
                _alpha=self.scheduler.alphas_cumprod[int(timestep)].to(
                    device=latents.device,dtype=torch.float32
                )
                _sa=torch.sqrt(_alpha.clamp_min(1e-12))
                _so=torch.sqrt((1.0-_alpha).clamp_min(1e-12))
                _xt=latents.float()
                _eu=noise_pred_uncon.float()
                _ec=noise_pred_con.float()
                _Du=(_xt-_so*_eu)/_sa
                _Dc=(_xt-_so*_ec)/_sa
                _delta_eps=_ec-_eu

                _beta=float(os.environ.get('FF_APG_MOMENTUM','-0.75'))
                if _apg_running is None:
                    _apg_running=_delta_eps.detach()
                else:
                    _apg_running=(_delta_eps+_beta*_apg_running).detach()

                _c_t=(_so/_sa).detach()
                _d=-_c_t*_apg_running

                _rbase=float(os.environ.get('FF_APG_NORM','5.0'))
                if _rbase>0:
                    _dn=torch.linalg.vector_norm(_d,ord=2,dim=(-3,-2,-1),keepdim=True)
                    _r_eff=_rbase*_c_t
                    _scale=torch.minimum(torch.ones_like(_dn),_r_eff/(_dn+1e-8))
                    _d=_d*_scale
                    if _apg_do_log:
                        self._ff_apg_telemetry_used=True
                        print(
                            f'[APG] t={int(timestep)} c={float(_c_t.cpu()):.5f} '
                            f'r_eff={float(_r_eff.cpu()):.5f} '
                            f'norm={float(_dn.mean().detach().cpu()):.5f} '
                            f'scale={float(_scale.mean().detach().cpu()):.5f}'
                        )

                _dot=(_d*_Dc).sum(dim=(-3,-2,-1),keepdim=True)
                _den=(_Dc*_Dc).sum(dim=(-3,-2,-1),keepdim=True).clamp_min(1e-8)
                _parallel=(_dot/_den)*_Dc
                _orth=_d-_parallel
                _eta=float(os.environ.get('FF_APG_ETA','0.0'))
                _mod=_orth+_eta*_parallel
                _Dg=_Dc+(float(guidance_scale)-1.0)*_mod
                _eg=(_xt-_sa*_Dg)/_so
                _eg=_eg.to(noise_pred_con.dtype)

                if not local_edit_text:
                    noise_pred=_eg
                else:
                    _m=completion_mask_cfg.to(noise_pred_con.dtype)
                    noise_pred=noise_pred_uncon+(_eg-noise_pred_uncon)*_m
            else:
                if not local_edit_text:
                    noise_pred=noise_pred_uncon+guidance_scale*(noise_pred_con-noise_pred_uncon)
                else:
                    local_text_guidance=guidance_scale*(noise_pred_con-noise_pred_uncon)*completion_mask_cfg
                    noise_pred=noise_pred_uncon+local_text_guidance'''

_lines=guide_new.splitlines()
_non=[ln for ln in _lines if ln.strip()]
_min=min(len(ln)-len(ln.lstrip(" ")) for ln in _non)
_norm="\n".join((_indent+ln[_min:]) if ln.strip() else "" for ln in _lines)
body=body[:_gm.start()]+_norm+body[_gm.end():]

m=pre+"    def forward_sampling("+body+"    def prox_regularization"+post

open(S,"w").write(s)
open(M,"w").write(m)
open(A,"w").write(a)

for _f in (S,M,A):
    py_compile.compile(_f,doraise=True)

# APG identity regression.
import torch
torch.manual_seed(42)
_xt=torch.randn(1,4,8,8); _eu=torch.randn_like(_xt); _ec=torch.randn_like(_xt)
_aa=torch.tensor(0.37); _sa=_aa.sqrt(); _so=(1-_aa).sqrt()
_Du=(_xt-_so*_eu)/_sa; _Dc=(_xt-_so*_ec)/_sa; _w=10.0
_Dg=_Dc+(_w-1.0)*(_Dc-_Du); _eg=(_xt-_sa*_Dg)/_so
_cfg=_eu+_w*(_ec-_eu)
assert float((_eg-_cfg).abs().max())<2e-5

print("✓ FINAL B patch installed + syntax-compiled")
print("✓ task switches + exact shift + AA + ring lock + HFF combos + EPSREC APG")


✓ replacement re-indented to 16 spaces
✓ forward_sampling UNet call patched via regex
✓ FINAL B patch installed + syntax-compiled
✓ task switches + exact shift + AA + ring lock + HFF combos + EPSREC APG


In [5]:

# ===== 5. Full GeoBench-2D manifest + deterministic four-account shard =====
SHARD_ID=1
N_SHARDS=4
EXPECTED_MANIFEST_SHA="17d971102daca232921d33de91adac58ef0ddbea9c9f814e219148486c9de802"

import os,glob,json,csv,shutil,hashlib,math
from collections import defaultdict,Counter

GEO="/kaggle/temp/GeoBenchMeta"
os.makedirs(f"{GEO}/Geo-Bench-2D",exist_ok=True)

def one(pattern,desc):
    xs=glob.glob(pattern,recursive=True)
    if not xs: raise FileNotFoundError(f"Missing {desc}: {pattern}")
    return sorted(xs,key=lambda x:(len(x),x))[0]

CACHE=next(c for c in glob.glob("/kaggle/input/**/Geo-Bench-2D",recursive=True)
           if os.path.isdir(f"{c}/source_img"))
COARSE=one("/kaggle/input/**/coarse_img/*/*/*.png","coarse_img").split("/coarse_img/")[0]+"/coarse_img"
IB=os.path.dirname(os.path.dirname(os.path.dirname(
    one("/kaggle/input/**/inp_img_blended/**/inp_img.png","inp_img_blended")
)))
ANNP=one("/kaggle/input/**/annotation_2d.json","annotation_2d.json")
META=one("/kaggle/input/**/sample_metadata.csv","sample_metadata.csv")
HIST_BASE=one("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup","historical full baseline")

for nm in ["source_img","source_mask","target_mask","source_img_full_v2"]:
    d=f"{GEO}/Geo-Bench-2D/{nm}"
    if os.path.lexists(d):
        os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
    os.symlink(f"{CACHE}/{nm}",d)
for nm,sc in [("coarse_img",COARSE),("inp_img_blended",IB)]:
    d=f"{GEO}/Geo-Bench-2D/{nm}"
    if os.path.lexists(d):
        os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
    os.symlink(sc,d)
shutil.copy(ANNP,f"{GEO}/annotation_2d.json")
ann=json.load(open(f"{GEO}/annotation_2d.json"))

meta=[]
for r in csv.DictReader(open(META)):
    if r["edit_type"] not in ("move","rotate","resize"): continue
    if not os.path.exists(f"{IB}/{r['da_n']}/{r['ins_id']}/inp_img.png"): continue
    d,i,e=str(r["da_n"]),str(r["ins_id"]),str(r["case_id"])
    ep=ann[d]["instances"][i][e]["edit_param"]
    dx,dy,dz,rx,ry,rz,sx,sy,sz=[float(x) for x in ep]
    scale=math.sqrt(sx*sy)
    severe=(r["edit_type"]=="resize" and (scale>=1.5 or scale<=0.6))
    rr=dict(r)
    rr.update({"da_n":d,"ins_id":i,"case_id":e,"scale_exact":scale,"affine_severe":severe})
    meta.append(rr)

counts=Counter(r["edit_type"] for r in meta)
assert len(meta)==5677,(len(meta),counts)
assert counts=={"resize":2635,"rotate":1603,"move":1439},counts
assert sum(r["affine_severe"] for r in meta)==759
assert sum(r["edit_type"]=="resize" and not r["affine_severe"] for r in meta)==1876

# Exact-affine manifest fingerprint used by Step 0 v2.
lines=[]
for r in sorted(meta,key=lambda z:(z["da_n"],z["ins_id"],z["case_id"])):
    ep=ann[r["da_n"]]["instances"][r["ins_id"]][r["case_id"]]["edit_param"]
    vals=[float(x) for x in ep]
    lines.append(
        f"{r['da_n']}|{r['ins_id']}|{r['case_id']}|{r['edit_type']}|"+
        "|".join(f"{x:.12g}" for x in vals)
    )
manifest_sha=hashlib.sha256("\n".join(lines).encode()).hexdigest()
assert manifest_sha==EXPECTED_MANIFEST_SHA,(manifest_sha,EXPECTED_MANIFEST_SHA)
print("✓ full exact-affine manifest",manifest_sha)

# Deterministic balanced shard assignment. Offsets distribute remainders so totals are:
# shard0=1419, shard1=1419, shard2=1420, shard3=1419
def category(r):
    if r["edit_type"]=="move": return "move"
    if r["edit_type"]=="rotate": return "rotate"
    return "resize_severe" if r["affine_severe"] else "resize_nonsevere"

offset={"move":0,"rotate":1,"resize_nonsevere":0,"resize_severe":2}
bycat=defaultdict(list)
for r in meta: bycat[category(r)].append(r)

assigned={s:[] for s in range(4)}
for cat,rows in bycat.items():
    rows=sorted(rows,key=lambda z:(z["da_n"],z["ins_id"],z["case_id"]))
    for j,r in enumerate(rows):
        s=(j+offset[cat])%4
        assigned[s].append(r)

expected_total=[1419,1419,1420,1419]
expected_severe=[190,189,190,190]
for s in range(4):
    assert len(assigned[s])==expected_total[s],(s,len(assigned[s]))
    assert sum(r["affine_severe"] for r in assigned[s])==expected_severe[s]

rows=sorted(assigned[SHARD_ID],key=lambda z:(z["da_n"],z["ins_id"],z["case_id"]))
severe_rows=[r for r in rows if r["affine_severe"]]
print("SHARD",SHARD_ID,"n=",len(rows),"severe=",len(severe_rows),
      "types=",dict(Counter(r["edit_type"] for r in rows)))

def build_manifest(rows,path):
    o={}
    for r in rows:
        d,i,e=r["da_n"],r["ins_id"],r["case_id"]
        lf=dict(ann[d]["instances"][i][e])
        lf["ori_img_path"]=os.path.join(GEO,lf["ori_img_path"])
        lf["ori_mask_path"]=os.path.join(GEO,lf["ori_mask_path"])
        o.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
    json.dump(o,open(path,"w"))
    return path

# Split each phase into disjoint one-GPU manifests.
for name,rr in [("baseline",rows),("severe",severe_rows)]:
    a=rr[::2]; b=rr[1::2]
    build_manifest(a,f"{GEO}/{name}_gpu0.json")
    build_manifest(b,f"{GEO}/{name}_gpu1.json")
    print(name,"gpu split",len(a),len(b))

json.dump(rows,open(f"{GEO}/shard_rows.json","w"),indent=2)
json.dump(severe_rows,open(f"{GEO}/shard_severe_rows.json","w"),indent=2)

# Output roots.
ROOT=f"/kaggle/working/final_geometry_full/shard_{SHARD_ID}"
BASE=f"{ROOT}/baseline"
A=f"{ROOT}/SGR_EPSREC"
B=f"{ROOT}/SGR_MIDHF_EPSREC"
for p in [BASE,A,B]: os.makedirs(p,exist_ok=True)

# Resume bootstrap: if a previous saved version of this shard is attached as input,
# copy the most complete previous tree into working. This never imports old experimental
# outputs; it only resumes this exact final-run shard.
def pc(p): return len(glob.glob(p+"/**/*.png",recursive=True)) if os.path.isdir(p) else 0
for leaf,dst in [("baseline",BASE),("SGR_EPSREC",A),("SGR_MIDHF_EPSREC",B)]:
    cands=glob.glob(f"/kaggle/input/**/final_geometry_full/shard_{SHARD_ID}/{leaf}",recursive=True)
    cands=[p for p in cands if os.path.realpath(p)!=os.path.realpath(dst)]
    if cands:
        src=max(cands,key=pc)
        if pc(src)>pc(dst):
            shutil.copytree(src,dst,dirs_exist_ok=True)
            print("resume bootstrap",leaf,pc(src),"from",src)

print("existing counts",{"baseline":pc(BASE),"A":pc(A),"B":pc(B)})


✓ full exact-affine manifest 17d971102daca232921d33de91adac58ef0ddbea9c9f814e219148486c9de802
SHARD 1 n= 1419 severe= 189 types= {'move': 360, 'rotate': 401, 'resize': 658}
baseline gpu split 710 709
severe gpu split 95 94
existing counts {'baseline': 0, 'A': 0, 'B': 0}


In [6]:

# ===== 6. Baseline no-op validation gate (3 fresh samples vs reproduced full baseline) =====
import os,json,socket,subprocess,glob,numpy as np,shutil
from PIL import Image

# HF token is optional for public SD-1.5; use it when configured.
try:
    from kaggle_secrets import UserSecretsClient
    tok=UserSecretsClient().get_secret("HF_TOKEN")
    if tok: os.environ["HF_TOKEN"]=tok
except Exception:
    pass

P="/kaggle/temp/FreeFine/evaluation/FreeFine"
rows=json.load(open(f"{GEO}/shard_rows.json"))
valrows=rows[:3]
build_manifest(valrows,f"{GEO}/val3.json")
VAL="/kaggle/temp/val3"
if os.path.exists(VAL): shutil.rmtree(VAL)
os.makedirs(VAL,exist_ok=True)

env=os.environ.copy()
env.update({
    "PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],
    "PYTHONUNBUFFERED":"1",
    "TOKENIZERS_PARALLELISM":"false",
    "HF_HOME":"/kaggle/temp/hf",
    "NCCL_P2P_DISABLE":"1",
    "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
    "FF_SUBSET_JSON":f"{GEO}/val3.json",
    "FF_OUT_DIR":VAL,
})
s=socket.socket(); s.bind(("",0)); port=s.getsockname()[1]; s.close()
r=subprocess.run(
    ["/kaggle/temp/freefine_env/bin/torchrun","--nproc_per_node=1",
     "--master-port",str(port),"freefine_sweep_2d.py"],
    cwd=P,env=env,capture_output=True,text=True
)
imgs=sorted(glob.glob(VAL+"/**/*.png",recursive=True))
if len(imgs)!=3:
    print((r.stdout+r.stderr)[-4000:])
    raise RuntimeError(f"validation generated {len(imgs)}/3")

diffs=[]
for p in imgs:
    rel=os.path.relpath(p,VAL)
    hp=f"{HIST_BASE}/{rel}"
    assert os.path.exists(hp),hp
    a=np.array(Image.open(p).convert("RGB"),dtype=float)
    b=np.array(Image.open(hp).convert("RGB").resize(Image.open(p).size),dtype=float)
    diffs.append(float(np.abs(a-b).mean()))
print("validation mean|Δ|",diffs)
assert max(diffs)<1.0, "Final environment does not reproduce baseline"
print("✓ baseline validation passed")


validation mean|Δ| [0.0, 0.0, 0.0]
✓ baseline validation passed


In [7]:

# ===== 7. Fresh final generation: baseline + exact common branches + two severe branches =====
import os,time,subprocess,socket,glob,json,shutil
import numpy as np, cv2
from PIL import Image

P="/kaggle/temp/FreeFine/evaluation/FreeFine"
HARD_DEADLINE=NB_START+11.45*3600

def count_png(p):
    return len(glob.glob(p+"/**/*.png",recursive=True)) if os.path.isdir(p) else 0

def free_port():
    s=socket.socket(); s.bind(("",0)); p=s.getsockname()[1]; s.close(); return p

def run_phase(tag, manifest_prefix, outdir, opts):
    """Run two disjoint one-GPU manifests. Resume-safe because FreeFine skips existing files."""
    if time.time()>HARD_DEADLINE-300:
        print(tag,"SKIP: deadline too close")
        return False

    procs=[]
    for gpu in [0,1]:
        man=f"{GEO}/{manifest_prefix}_gpu{gpu}.json"
        env=os.environ.copy()
        env.update({
            "CUDA_VISIBLE_DEVICES":str(gpu),
            "PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],
            "PYTHONUNBUFFERED":"1",
            "TOKENIZERS_PARALLELISM":"false",
            "HF_HOME":"/kaggle/temp/hf",
            "NCCL_P2P_DISABLE":"1",
            "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
            "FF_SUBSET_JSON":man,
            "FF_OUT_DIR":outdir,
        })
        env.update({k:str(v) for k,v in opts.items()})
        logp=f"{ROOT}/{tag}_gpu{gpu}.log"
        log=open(logp,"a")
        proc=subprocess.Popen(
            ["/kaggle/temp/freefine_env/bin/torchrun","--nproc_per_node=1",
             "--master-port",str(free_port()),"freefine_sweep_2d.py"],
            cwd=P,env=env,stdout=log,stderr=subprocess.STDOUT
        )
        procs.append((gpu,proc,log,logp))
        print(f"[{tag}] GPU{gpu} launched",flush=True)

    while any(p.poll() is None for _,p,_,_ in procs):
        if time.time()>HARD_DEADLINE:
            print(tag,"HARD DEADLINE — terminating cleanly",flush=True)
            for _,p,_,_ in procs:
                if p.poll() is None: p.terminate()
            time.sleep(5)
            for _,p,_,_ in procs:
                if p.poll() is None: p.kill()
            break
        time.sleep(20)

    ok=True
    for gpu,p,log,logp in procs:
        rc=p.poll()
        log.close()
        if rc not in (0,None):
            ok=False
            try: print(open(logp).read()[-3500:])
            except Exception: pass
    print(tag,"output pngs=",count_png(outdir))
    return ok

# Phase 1: fresh baseline for every sample in this shard.
run_phase("baseline","baseline",BASE,{})

rows=json.load(open(f"{GEO}/shard_rows.json"))
severe_rows=json.load(open(f"{GEO}/shard_severe_rows.json"))
expected=len(rows); severe_n=len(severe_rows)

# Exact common branches are part of the final method:
# move -> RING4 post-lock
# rotate -> baseline
# resize non-severe -> RING8 post-lock
# Severe resize is intentionally left empty here and filled by A/B diffusion branches.
def imgp(root,r):
    return f"{root}/{r['da_n']}/{r['ins_id']}/{r['case_id']}.png"

def save_common(dstroot,r):
    bp=imgp(BASE,r)
    if not os.path.exists(bp): return False
    dst=imgp(dstroot,r)
    os.makedirs(os.path.dirname(dst),exist_ok=True)
    if r["affine_severe"]:
        return False
    if r["edit_type"]=="rotate":
        shutil.copy2(bp,dst)
        return True
    width=4 if r["edit_type"]=="move" else 8
    cp=f"{GEO}/Geo-Bench-2D/coarse_img/{r['da_n']}/{r['ins_id']}/{r['case_id']}.png"
    tp=f"{GEO}/Geo-Bench-2D/target_mask/{r['da_n']}/{r['ins_id']}/{r['case_id']}.png"
    G=np.array(Image.open(bp).convert("RGB"))
    C=np.array(Image.open(cp).convert("RGB"))
    T=np.array(Image.open(tp).convert("L"))>127
    interior=cv2.erode(
        T.astype(np.uint8),
        np.ones((2*width+1,2*width+1),np.uint8),
        iterations=1
    ).astype(bool)
    O=np.where(interior[:,:,None],C,G).astype(np.uint8)
    Image.fromarray(O).save(dst)
    return True

common_done=0
for r in rows:
    if r["affine_severe"]: continue
    if not os.path.exists(imgp(A,r)):
        save_common(A,r)
    if not os.path.exists(imgp(B,r)):
        save_common(B,r)
    if os.path.exists(imgp(A,r)) and os.path.exists(imgp(B,r)):
        common_done+=1

print("common branches complete",common_done,"/",expected-severe_n)

# Phase 2: SGR-EPSREC severe resize, exact measured EPSREC_PROMPT settings.
APG={
    "FF_APG":1,
    "FF_GUIDANCE_MODE":"apg_exact",
    "FF_APG_ETA":0.0,
    "FF_APG_NORM":5.0,
    "FF_APG_MOMENTUM":-0.75,
    "FF_APG_TYPES":"resize",
    "FF_PROMPT_TYPES":"resize",
    "FF_PROMPT_GUIDANCE":10,
}
if count_png(BASE)>=expected:
    run_phase("severe_A_EPSREC","severe",A,APG)

# Phase 3: SGR-MIDHF+EPSREC severe resize, exact measured MIDHF settings.
MIDHF={
    **APG,
    "FF_HFF":1,
    "FF_HFF_TYPES":"resize",
    "FF_HFF_BETA":0.30,
    "FF_HFF_HORIZON":0.55,
    "FF_HFF_RADIUS":3,
    "FF_HFF_BLOCK":1,
    "FF_HFF_MODE":"midhf",
    "FF_HFF_MID_LOW":1,
    "FF_HFF_MID_HIGH":3,
    "FF_HFF_HIGH_WEIGHT":0.65,
}
if count_png(BASE)>=expected:
    run_phase("severe_B_MIDHF_EPSREC","severe",B,MIDHF)

# Final completeness audit.
def route_counts(root):
    ok=Counter()
    miss=[]
    for r in rows:
        p=imgp(root,r)
        if os.path.exists(p): ok[category(r)]+=1
        else: miss.append((r["da_n"],r["ins_id"],r["case_id"],category(r)))
    return ok,miss

base_n=count_png(BASE)
ca,ma=route_counts(A)
cb,mb=route_counts(B)

status={
    "shard_id":SHARD_ID,
    "expected":expected,
    "expected_severe":severe_n,
    "baseline_pngs":base_n,
    "pipeline_A_pngs":count_png(A),
    "pipeline_B_pngs":count_png(B),
    "pipeline_A_route_counts":dict(ca),
    "pipeline_B_route_counts":dict(cb),
    "pipeline_A_missing":ma[:100],
    "pipeline_B_missing":mb[:100],
    "manifest_sha256":manifest_sha,
    "complete":(
        base_n>=expected and count_png(A)>=expected and count_png(B)>=expected
        and not ma and not mb
    )
}
json.dump(status,open(f"{ROOT}/generation_status.json","w"),indent=2)
json.dump(rows,open(f"{ROOT}/shard_manifest.json","w"),indent=2)

print(json.dumps({k:v for k,v in status.items() if k not in ("pipeline_A_missing","pipeline_B_missing")},indent=2))
if status["complete"]:
    print("\n✓ FINAL SHARD COMPLETE — Save Version and keep this output.")
else:
    print("\nPARTIAL FINAL SHARD — Save Version, attach this saved output as a Notebook Input, and rerun the same shard notebook.")


[baseline] GPU0 launched
[baseline] GPU1 launched
baseline output pngs= 1419
common branches complete 1230 / 1230
[severe_A_EPSREC] GPU0 launched
[severe_A_EPSREC] GPU1 launched
severe_A_EPSREC output pngs= 1419
[severe_B_MIDHF_EPSREC] GPU0 launched
[severe_B_MIDHF_EPSREC] GPU1 launched
severe_B_MIDHF_EPSREC output pngs= 1419
{
  "shard_id": 1,
  "expected": 1419,
  "expected_severe": 189,
  "baseline_pngs": 1419,
  "pipeline_A_pngs": 1419,
  "pipeline_B_pngs": 1419,
  "pipeline_A_route_counts": {
    "move": 360,
    "rotate": 401,
    "resize_nonsevere": 469,
    "resize_severe": 189
  },
  "pipeline_B_route_counts": {
    "move": 360,
    "rotate": 401,
    "resize_nonsevere": 469,
    "resize_severe": 189
  },
  "manifest_sha256": "17d971102daca232921d33de91adac58ef0ddbea9c9f814e219148486c9de802",
  "complete": true
}

✓ FINAL SHARD COMPLETE — Save Version and keep this output.
